<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/05c_mitre_atlas_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5c: MITRE ATLAS v5.4.0 Technique Mapping and NIST AI RMF Compliance Report Card

**Goal:** Map Phase 5a (OWASP LLM Top 10) and Phase 5b (OWASP Agentic Top 10) attack results onto MITRE ATLAS v5.4.0 techniques using a deterministic lookup table, not a model judgment call, since the OWASP-category-to-ATLAS-technique relationship is a fixed taxonomy, not something requiring case-by-case interpretation. Generate a NIST AI RMF compliance report card summarizing detection coverage.

**Design decision, stated plainly:** unlike 05a/05b's attack detection logic, this notebook's core mapping requires no API call at all, live or simulated. A lookup table is not a placeholder for a future real implementation, it is the correct, permanent implementation. This is why this notebook has no `SIMULATED_OUTPUT` flag for the mapping step itself.

**Honest sourcing caveat:** there is no single official crosswalk published jointly by OWASP and MITRE. The mappings below are synthesized from MITRE's own published ATLAS technique descriptions, cross-checked against multiple independent security-industry writeups, not copied from one authoritative source. Each mapping below is labeled by confidence: **direct** (name and scope closely match, corroborated across multiple sources), **interpretive** (a reasonable synthesis, not universally published this way), or **unresolved** (no clean existing technique identified). This labeling is itself part of the deliverable, not a hedge to remove later.

**Tools:** MITRE ATLAS v5.4.0 technique taxonomy (AML.T-prefixed technique IDs), NIST AI RMF

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm Phase 5a and 5b

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase5a_path = DRIVE_PATH + "phase05a_promptfoo_owasp_results.json"
phase5b_path = DRIVE_PATH + "phase05b_promptfoo_owasp_agentic_results.json"

missing = []
if os.path.exists(phase5a_path):
    with open(phase5a_path) as f:
        phase5a = json.load(f)
    print("Phase 5a results confirmed.")
    print(f"  Detection rate: {phase5a['detection_rate']:.0%} "
          f"({phase5a['detected_count']}/{phase5a['attack_case_count']})")
else:
    missing.append(phase5a_path)

if os.path.exists(phase5b_path):
    with open(phase5b_path) as f:
        phase5b = json.load(f)
    print("Phase 5b results confirmed.")
    print(f"  Detection rate: {phase5b['detection_rate']:.0%} "
          f"({phase5b['detected_count']}/{phase5b['attack_case_count']})")
else:
    missing.append(phase5b_path)

if missing:
    print("WARNING: missing files:")
    for m in missing:
        print(" ", m)
    print("Run 05a_promptfoo_owasp_llm.ipynb and/or 05b_promptfoo_owasp_agentic.ipynb first.")

Mounted at /content/drive
Phase 5a results confirmed.
  Detection rate: 100% (10/10)
Phase 5b results confirmed.
  Detection rate: 100% (12/12)


In [2]:
# Cell 3: Install packages
# This notebook's core work, the ATLAS mapping and the NIST report card, is
# deterministic and needs no LLM API at all. The only install here is
# Langfuse, kept for consistency with every other phase's trace logging,
# and pandas, used only to format the report card table.

!pip install langfuse pandas --quiet

print("Packages installed.")
print("No Gemini or Claude client is needed in this notebook's core logic.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.
No Gemini or Claude client is needed in this notebook's core logic.
